To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

Long-Context GRPO for reinforcement learning — train stably at massive sequence lengths. Fine-tune models with up to 7x more context length efficiently. [Read Blog](https://unsloth.ai/docs/new/grpo-long-context)

3× faster training with optimized sequence packing — higher throughput with no quality loss.[Read Blog](https://unsloth.ai/docs/new/3x-faster-training-packing)

500k context-length fine-tuning — push long-context models further with memory-efficient training. [Read Blog](https://unsloth.ai/docs/new/500k-context-length-fine-tuning)

Introducing FP8 precision training for faster RL inference. [Read Blog](https://docs.unsloth.ai/new/fp8-reinforcement-learning).

Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://docs.unsloth.ai/new/how-to-train-llms-with-unsloth-and-docker).

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install torchao==0.14.0 executorch pytorch_tokenizers

### Unsloth

In [20]:
from unsloth import FastLanguageModel
import torch

# Models supported for Phone Deployment
fourbit_models = [
    "unsloth/Qwen3-4B",              # Any Qwen3 model like 0.6B, 4B, 8B, 32B
    "unsloth/Qwen3-32B",
    "unsloth/Llama-3.1-8B-Instruct", # Llama 3 models work
    "unsloth/Llama-3.3-70B-Instruct",
    "unsloth/gemma-3-270m-it",       # Gemma 3 models work
    "unsloth/gemma-3-27b-it",
    "unsloth/Qwen2.5-7B-Instruct",   # And more models!
    "unsloth/Phi-4-mini-instruct",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-1.7B",
    max_seq_length = 4096,
    full_finetuning = True,
    qat_scheme = "phone-deployment"
)

Unsloth: You selected full finetuning support, but 4bit / 8bit is enabled - disabling LoRA / QLoRA.
==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Float16 full finetuning uses more memory since we upcast weights to float32.


model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Unsloth: Applying QAT to mitigate quantization degradation


<a name="Save"></a>
### Saving, loading finetuned models

To save the model for phone deployment, we first save the model via `save_pretrained_torchao`

In [22]:
model.save_pretrained_torchao("qwen3_fp16", tokenizer)
#tokenizer.save_pretrained("qwen3_model_fp16")

In [15]:
import safetensors.torch as st

state = st.load_file("qwen3_fp16/model.safetensors")

bad = [k for k in state.keys() if "absmax" in k or "scale" in k]
print(bad)

['model.layers.0.mlp.down_proj.weight.nested_absmax', 'model.layers.0.mlp.gate_proj.weight.nested_absmax', 'model.layers.0.mlp.up_proj.weight.nested_absmax', 'model.layers.0.self_attn.k_proj.weight.nested_absmax', 'model.layers.0.self_attn.o_proj.weight.nested_absmax', 'model.layers.0.self_attn.q_proj.weight.nested_absmax', 'model.layers.0.self_attn.v_proj.weight.nested_absmax', 'model.layers.1.mlp.down_proj.weight.nested_absmax', 'model.layers.1.mlp.gate_proj.weight.nested_absmax', 'model.layers.1.mlp.up_proj.weight.nested_absmax', 'model.layers.1.self_attn.k_proj.weight.nested_absmax', 'model.layers.1.self_attn.o_proj.weight.nested_absmax', 'model.layers.1.self_attn.q_proj.weight.nested_absmax', 'model.layers.1.self_attn.v_proj.weight.nested_absmax', 'model.layers.10.mlp.down_proj.weight.nested_absmax', 'model.layers.10.mlp.gate_proj.weight.nested_absmax', 'model.layers.10.mlp.up_proj.weight.nested_absmax', 'model.layers.10.self_attn.k_proj.weight.nested_absmax', 'model.layers.10.sel

We then use Executorch's Qwen3 conversion process

In [23]:
# Convert the weight checkpoint state dict keys to one that ExecuTorch expects
!python -m executorch.examples.models.qwen3.convert_weights \
    "qwen3_fp16" pytorch_model_converted.bin

Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu126 for torchao version 0.14.0         Please see GitHub issue #2919 for more info
I tokenizers:regex.cpp:27] Registering override fallback regex
<frozen runpy>:128: RuntimeWarning: 'executorch.examples.models.qwen3.convert_weights' found in sys.modules after import of package 'executorch.examples.models.qwen3', but prior to execution of 'executorch.examples.models.qwen3.convert_weights'; this may result in unpredictable behaviour
Loading checkpoint...
Loading checkpoint from pytorch_model directory
Converting checkpoint...
Saving checkpoint...
Done.


In [24]:
# Download model config from ExecuTorch repo
!curl -L -o 1.7B_config.json https://raw.githubusercontent.com/pytorch/executorch/main/examples/models/qwen3/config/1_7b_config.json

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   348  100   348    0     0    875      0 --:--:-- --:--:-- --:--:--   876


And finally we export to a .pte file which can be used for deployment

In [31]:
# Export to ExecuTorch pte file
!python -m executorch.examples.models.llama.export_llama \
    --model "qwen3_1_7b" \
    --checkpoint pytorch_model_converted.bin \
    --params 1.7B_config.json \
    --output_name qwen3_1.7B_model.pte \
    -kv \
    --use_sdpa_with_kv_cache \
    --coreml \
    --coreml-ios 16 \
    --coreml-compute-units all \
    --max_context_length 4096 \
    --max_seq_length 4096 \
    --disable_dynamic_shape \
    --dtype fp16 \
    --metadata '{"get_bos_id":199999, "get_eos_ids":[200020,199999]}'

Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu126 for torchao version 0.14.0         Please see GitHub issue #2919 for more info
I tokenizers:regex.cpp:27] Registering override fallback regex
[INFO 2026-02-03 12:58:25,900 utils.py:164] NumExpr defaulting to 2 threads.
[INFO 2026-02-03 12:58:26,340 export_llama_lib.py:781] Applying quantizers: []
[WARNING 2026-02-03 12:58:28,372 export_llama_lib.py:695] Checkpoint dtype torch.float32 precision is higher than dtype override torch.float16.
[INFO 2026-02-03 12:58:30,300 export_llama_lib.py:703] Checkpoint dtype: torch.float32
[INFO 2026-02-03 12:58:30,300 custom_kv_cache.py:367] Replacing KVCache with CustomKVCache. This modifies the model in place.
[INFO 2026-02-03 12:58:30,385 custom_ops.py:34] Looking for libcustom_ops_aot_lib.so in /usr/local/lib/python3.12/dist-packages/executorch/extension/llm/custom_ops
[INFO 2026-02-03 12:58:30,389 custom_ops.py:39] Loading custom ops library: /usr/local/lib/python3.12

And we have the file `qwen3_0.6B_model.pte` of around 472MB!

In [ ]:
!ls -l -h qwen3_1.7B_model.pte

-rw-r--r-- 1 root root 1.2G Jan 24 13:06 qwen3_1.7B_model.pte


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# This tells the shell to copy everything except the 'drive' directory
!shopt -s extglob
!cp -r !(drive) /content/drive/MyDrive/z_temp/qwen3

Mounted at /content/drive
^C


And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).
